# HOG Feature Visualization

This notebook visualizes how Histogram of Oriented Gradients (HOG) represents image edges before the SVM classifier uses those features.

In [ ]:
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
from skimage.exposure import rescale_intensity
from skimage.feature import hog

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.append(str(PROJECT_ROOT / "src"))

from config import CLASS_LABELS, HOG_CONFIG, IMAGE_SIZE, TRAIN_DIR

plt.style.use("default")

## Load A Few Training Images

In [ ]:
image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}
sample_paths = []

for class_name in CLASS_LABELS:
    class_dir = Path(TRAIN_DIR) / class_name
    class_images = sorted(
        path for path in class_dir.glob("*")
        if path.suffix.lower() in image_extensions
    )
    sample_paths.extend(class_images[:2])

sample_paths = sample_paths[:4]
print(f"Found {len(sample_paths)} sample images")
for path in sample_paths:
    print(path.relative_to(PROJECT_ROOT))

## Original, Grayscale, And HOG Image

In [ ]:
def load_rgb_and_gray(image_path):
    bgr = cv2.imread(str(image_path))
    if bgr is None:
        raise ValueError(f"Could not load image: {image_path}")

    bgr = cv2.resize(bgr, IMAGE_SIZE)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    return rgb, gray


def extract_hog_with_image(gray):
    features, hog_image = hog(
        gray,
        orientations=HOG_CONFIG["orientations"],
        pixels_per_cell=HOG_CONFIG["pixels_per_cell"],
        cells_per_block=HOG_CONFIG["cells_per_block"],
        visualize=True,
        block_norm=HOG_CONFIG["block_norm"],
        transform_sqrt=HOG_CONFIG["transform_sqrt"],
    )
    hog_image = rescale_intensity(hog_image, in_range=(0, 10))
    return features, hog_image

In [ ]:
if not sample_paths:
    print("No images found. Add training images under data/raw/train/class_0 and data/raw/train/class_1.")
else:
    fig, axes = plt.subplots(len(sample_paths), 3, figsize=(10, 3 * len(sample_paths)))
    if len(sample_paths) == 1:
        axes = np.array([axes])

    feature_lengths = []
    for row, image_path in enumerate(sample_paths):
        rgb, gray = load_rgb_and_gray(image_path)
        features, hog_image = extract_hog_with_image(gray)
        feature_lengths.append(len(features))

        class_name = image_path.parent.name
        axes[row, 0].imshow(rgb)
        axes[row, 0].set_title(f"Original: {class_name}")
        axes[row, 1].imshow(gray, cmap="gray")
        axes[row, 1].set_title("Grayscale")
        axes[row, 2].imshow(hog_image, cmap="gray")
        axes[row, 2].set_title("HOG visualization")

        for ax in axes[row]:
            ax.axis("off")

    plt.tight_layout()
    plt.show()
    print(f"HOG feature vector length: {feature_lengths[0]}")

## HOG Feature Vector Preview

In [ ]:
if sample_paths:
    _, gray = load_rgb_and_gray(sample_paths[0])
    features, _ = extract_hog_with_image(gray)

    plt.figure(figsize=(12, 3))
    plt.plot(features[:300])
    plt.title("First 300 HOG Feature Values")
    plt.xlabel("Feature index")
    plt.ylabel("Value")
    plt.tight_layout()
    plt.show()

    print(f"Min: {features.min():.4f}")
    print(f"Max: {features.max():.4f}")
    print(f"Mean: {features.mean():.4f}")